# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 2: Prompt Template Engineering</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know how a base model becomes a chat model and why it stops after its answer, <br>
        ... can write raw chat prompts with special tokens, including the system prompt, <br>
        ... understand how reasoning models write their reasoning between <code>&lt;think&gt;</code> and <code>&lt;/think&gt;</code>, <br>
        ... can find a model's template and see the text that chat APIs build from your messages.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

Documentation for this notebook:

* [Hugging Face: Chat templates](https://huggingface.co/docs/transformers/chat_templating)
* [Ollama API documentation](https://docs.ollama.com/api)

## Where We Are

* **[4.01](https://github.com/olivermueller/amlta-2026/blob/main/Session_04/4.01%20-%20Basics%20of%20Prompt%20Engineering.ipynb):** a base model is a **continuation machine**. It does not stop on its own, and a stop sequence is a crude fix.
* **Now:** the better fix. **Chat models** were trained on one fixed **chat template** and end every answer with a special token.

We write these templates by hand, add a system prompt, and look at reasoning models.

## Setup

Same setup as in 4.01.

* **New Colab runtime:** run all cells of this section.
* **macOS / Windows with the Ollama app:** skip Steps 1 and 2.

### Step 1: Install Ollama

In [ ]:
!sudo apt-get install -y pciutils zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

### Step 2: Start the Ollama server

In [ ]:
import subprocess
import time

# the server writes its log into a file; a pipe that nobody reads could fill up and block the server
ollama_log = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)

# give the server a few seconds to start
time.sleep(5)

### Step 3: Pull the chat model

* **4.01:** `llama3.2:1b-text-q4_K_M`, the base model.
* **Now:** `llama3.2:1b`, the same model after additional training on conversations.

The reasoning model follows in Part 3.

In [ ]:
# You can work on a small gpu (e.g., T4) --> download will take a while (1 minute)
!ollama pull llama3.2:1b

In [ ]:
CHAT_MODEL = "llama3.2:1b"  # chat model: Llama 3.2 1B after additional training on conversations

### Step 4: Connect from Python

As in 4.01, we use Ollama's own Python package, `ollama`.

In [ ]:
!pip install ollama

In [ ]:
import ollama

# address of our local Ollama server
OLLAMA_HOST = "http://127.0.0.1:11434"

# Ollama's own client, as in 4.01
olm = ollama.Client(host=OLLAMA_HOST)

## Part 1: Chat Models

### From base model to chat model

![Foundation Models VS Chat Models](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_04/imgs/llms.png)

(Image adapted from ([Raschka (2024)](https://www.manning.com/books/build-a-large-language-model-from-scratch)))

**In short: a chat model is a base model that was fine-tuned on conversations.**

* **Pretraining** gives you a **foundation model**, also called **base model**, like the models you trained in the last session.
* **Fine-tuning** on conversations teaches it to follow instructions and to answer like a chat partner ([Ouyang et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/b1efde53be364a73914f58805a001731-Paper-Conference.pdf)). ChatGPT and Claude were made this way ([OpenAI, 2022](https://openai.com/index/chatgpt/); [Ganguli et al., 2022](https://doi.org/10.48550/arXiv.2209.07858)).

Our `CHAT_MODEL` is the same Llama 3.2 1B, fine-tuned on conversations. We fine-tune models ourselves in Session 7.

### How the model sees it

During fine-tuning, every conversation is written in one fixed format: the **chat template**. For Llama 3.2, it looks like this:

<pre>
<b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#1F5F9E">user</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#1F5F9E">What is the capital of France?</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b><b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#0B6048">assistant</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#0B6048">The capital of France is Paris.</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b>
</pre>

<b><font color="#B8860B">■ special token</font></b> &nbsp; <b><font color="#1F5F9E">■ user message</font></b> &nbsp; <b><font color="#0B6048">■ assistant (written by the model)</font></b> &nbsp; ↵ = line break

The parts:

* `<|start_header_id|>` and `<|end_header_id|>` enclose the **role**: `system`, `user`, or `assistant`.
* `<|eot_id|>` ("end of turn") closes every message.
* These are **special tokens**: each one is a single token that never appears in normal text.
* The text ends with an **open assistant header**, the so-called **generation prompt**. The model continues from there: it writes its answer and then `<|eot_id|>`. Ollama stops at this token and doesn't print it.
* Ollama adds the start token `<|begin_of_text|>` automatically, so we don't write it.

Let's write this by hand. First, we need a new tool.

### New: raw mode

* **In 4.01**, Ollama added nothing to our text, because the base model's template is just `{{ .Prompt }}`.
* **A chat model** has a real template. Ollama would wrap our hand-written text in a second set of special tokens.
* **The fix:** `raw=True` switches the template off, so the model sees exactly our text.

Two helpers in the next cell:

* `complete()` sends a raw prompt and prints the result: the model's continuation in green, special tokens highlighted. At the end, it tells you how many tokens were generated and why the model stopped.
* `score()` works as in 4.01: it fills `{text}` with every example and counts the correct answers.

In [ ]:
import re

GREEN, ORANGE, GREY, RESET = "\033[92m", "\033[33m", "\033[90m", "\033[0m"
SPECIAL_TOKENS = re.compile(r"(<\|[^|>]+\|>|</?think>|<start_of_turn>|<end_of_turn>)")


def highlight(text, color="", newlines=False):
    """Highlights special tokens (inverse video) and, if wanted, shows line breaks as ↵."""
    if newlines:
        text = text.replace("\n", "↵\n")
    text = SPECIAL_TOKENS.sub(lambda m: f"\033[7m{m.group(0)}\033[27m", text)
    return f"{color}{text}{RESET}"


def generate(prompt, model=CHAT_MODEL, max_tokens=80, temperature=0, seed=42, stop=None, **options):
    """Sends a raw prompt: no template, the model sees exactly this text and continues it."""
    options = {"num_predict": max_tokens, "temperature": temperature, "seed": seed, **options}
    if stop:
        options["stop"] = stop
    return olm.generate(model=model, prompt=prompt, raw=True, options=options)


def complete(prompt, model=CHAT_MODEL, max_tokens=80, temperature=0, seed=42, stop=None, newlines=False, **options):
    """Sends a raw prompt and prints it: our prompt in normal color, the continuation in green."""
    response = generate(prompt, model, max_tokens, temperature, seed, stop, **options)
    print(highlight(prompt, newlines=newlines), end="")
    if response.thinking:  # in case Ollama separated the reasoning of a reasoning model
        print(highlight("<think>\n" + response.thinking + "\n</think>\n\n", ORANGE, newlines), end="")
    print(highlight(response.response, GREEN, newlines))
    print(f"\n{GREY}[{response.eval_count or 0} tokens generated, stopped because of: {response.done_reason}]{RESET}")


def check(answer, expected):
    """Looks for the expected answer behind "The answer is" or, if the model did not write that, in the first line."""
    answer = answer.split("</think>")[-1]  # ignore the reasoning of reasoning models
    position = answer.lower().find("answer is")
    if position >= 0:
        answer = answer[position + len("answer is"):]
    first_line = next((line.strip() for line in answer.splitlines() if line.strip()), "")
    correct = re.search(rf"\b{re.escape(expected)}\b", first_line, re.IGNORECASE) is not None
    return correct, first_line


def score(template, examples, model=CHAT_MODEL, max_tokens=10, temperature=0, seed=42, stop=None, **options):
    """Fills {text} in the template with every example, sends it raw, and counts the correct answers."""
    print(f"{GREY}Prompt for the first example:{RESET}")
    print(highlight(template.replace("{text}", examples[0][0]), GREY) + "\n")
    correct, tokens = 0, 0
    for text, expected in examples:
        response = generate(template.replace("{text}", text), model, max_tokens, temperature, seed, stop, **options)
        ok, answer = check(response.response, expected)
        correct += ok
        tokens += response.eval_count or 0  # an empty prompt generates nothing
        mark = f"{GREEN}✓{RESET}" if ok else f"{ORANGE}✗{RESET}"
        print(f"{mark} expected: {expected:<10} model: {answer[:70]}")
    print(f"\n{correct}/{len(examples)} correct, {tokens} tokens generated")

Now we send our hand-written chat prompt to the chat model:

In [ ]:
chat_prompt = """<|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(chat_prompt, model=CHAT_MODEL, max_tokens=100, newlines=True)

`complete()` reports `stop`: the model generated `<|eot_id|>` on its own.

**The model doesn't know that it is the assistant.** It still just continues a text, like the base model. But during training it saw thousands of texts in this format, where an assistant answers and then writes `<|eot_id|>`. So it *imitates* an assistant, and the generation stops at that token.

### Your turn

1. **Predict:** what happens if the text ends with an open **user** header instead of an assistant header?
2. **Predict:** what happens with two user messages in a row? What does the model answer, and where does it get the name from?
3. Write a two-turn conversation by hand: a question, your own answer, a follow-up question.
4. A real task, information extraction: see below.

In [ ]:
open_user = """<|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of Germany?
<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of Italy?
<|eot_id|><|start_header_id|>user<|end_header_id|>
"""
complete(open_user, model=CHAT_MODEL, max_tokens=60, newlines=True)

In [ ]:
two_users = """<|start_header_id|>user<|end_header_id|>

My name is Anna.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is my name?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(two_users, model=CHAT_MODEL, max_tokens=60, newlines=True)

<details>
<summary><b>Explanation</b> (open after running the cells)</summary>

1. The model writes the **user's** message. It continues whatever role is open and has no idea which side it is on.
2. Look for "Anna". Both messages are in the same text, so the model can use the name. **A chat model has no memory: everything it "remembers" is in the text it receives.** Two user messages in a row are unusual, and our small model gets confused: it may use the name and still claim not to know it.
</details>

In [ ]:
# TODO: write a conversation with two turns by hand
my_conversation = """<|start_header_id|>user<|end_header_id|>

What is the largest planet in our solar system?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The largest planet in our solar system is Jupiter.<|eot_id|><|start_header_id|>user<|end_header_id|>

How long does it take to go around the sun?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(my_conversation, model=CHAT_MODEL, max_tokens=100, newlines=True)

4. **A real task: information extraction.** Extract the city where an event takes place. Some announcements mention more than one city.
    * Run the zero-shot template and read the answers.
    * Add your own example as an earlier `user` and `assistant` turn (few-shot inside a chat template). Does it help?

In [ ]:
# eight short announcements with the city in which the event takes place
EVENTS = [
    ("Our team meeting next Monday will be held in Munich instead of Cologne.", "Munich"),
    ("The concert, originally planned for Leipzig, has been moved to Dresden.", "Dresden"),
    ("Join us for the summer party in Paderborn on Friday at 6 pm!", "Paderborn"),
    ("The workshop takes place in Frankfurt. Participants from Bonn can take the train.", "Frankfurt"),
    ("The award ceremony will be held in Dortmund, not in Bremen as announced last week.", "Dortmund"),
    ("Students from Hanover and Kassel will meet in Erfurt for the hackathon.", "Erfurt"),
    ("After long discussions, the board decided to hold the annual meeting in Nuremberg.", "Nuremberg"),
    ("The press conference was moved from Berlin to Potsdam at short notice.", "Potsdam"),
]

In [ ]:
extraction_zero_shot = """<|start_header_id|>user<|end_header_id|>

In which city does the event take place? Answer with the city only.

{text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
score(extraction_zero_shot, EVENTS)

In [ ]:
# TODO: add an example as an earlier user and assistant turn (few-shot inside the chat template)
extraction_few_shot = """<|start_header_id|>user<|end_header_id|>

In which city does the event take place? Answer with the city only.

The book fair, first planned in Hamburg, now takes place in Stuttgart.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Stuttgart<|eot_id|><|start_header_id|>user<|end_header_id|>

In which city does the event take place? Answer with the city only.

{text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
score(extraction_few_shot, EVENTS)

### The easy way: messages as JSON

Nobody wants to write special tokens by hand. Chat APIs take a **list of messages**, each with a `role` and a `content`, and apply the template for you:

In [ ]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},  # roles: system, user, assistant
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What is the capital of Germany?"},  # roles: system, user, assistant
]

response = olm.chat(model=CHAT_MODEL, messages=messages, options={"temperature": 0, "seed": 42})
print(response.message.content)

What did the API send to the model? Ollama's debugging feature `_debug_render_only` builds the text from the messages without generating anything. Our last helper, `what_the_model_sees()`, uses it:

In [ ]:
import requests

def what_the_model_sees(model, messages, think=None, newlines=True):
    """Lets Ollama build the prompt from a list of messages, without generating anything."""
    payload = {"model": model, "messages": messages, "stream": False, "_debug_render_only": True}
    if think is not None:
        payload["think"] = think
    response = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    text = response.json()["_debug_info"]["rendered_template"]
    print(highlight(text, newlines=newlines))

In [ ]:
what_the_model_sees(CHAT_MODEL, messages)

Look at the top: the template adds a `system` block with `Cutting Knowledge Date: December 2023`, even though we didn't write a system message. **A template can add text that nobody wrote.**

And here are the two user messages from above, built by the API:

In [ ]:
what_the_model_sees(CHAT_MODEL, [
    {"role": "user", "content": "My name is Anna."},
    {"role": "user", "content": "What is my name?"},
])

## Part 2: The System Prompt

**In short: the system prompt is the first message. It holds the general instructions.**

* It has the role `system` and comes before everything else.
* It contains things like a role, a tone, rules, or an output format ([Zhang et al., 2024](https://doi.org/10.48550/arXiv.2410.14826)).
* It is written by the developer of an application, not by the user.

### How the model sees it

<pre>
<b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#7B3FA0">system</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#7B3FA0">You are an AI chatbot giving sarcastic answers.</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b><b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#1F5F9E">user</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#1F5F9E">How are you?</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b><b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#0B6048">assistant</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#0B6048">Oh, just living the dream of answering questions all day.</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b>
</pre>

<b><font color="#B8860B">■ special token</font></b> &nbsp; <b><font color="#7B3FA0">■ system prompt</font></b> &nbsp; <b><font color="#1F5F9E">■ user message</font></b> &nbsp; <b><font color="#0B6048">■ assistant (written by the model)</font></b> &nbsp; ↵ = line break

To the model, the system prompt is just the first part of the text. During training it learned to give this part extra weight, but it is still text, not a rule the model has to obey.

In [ ]:
system_prompt = """<|start_header_id|>system<|end_header_id|>

You are an AI chatbot giving sarcastic answers.<|eot_id|><|start_header_id|>user<|end_header_id|>

How are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(system_prompt, model=CHAT_MODEL, max_tokens=100, newlines=True)

### Your turn: answers in Morse code

Write a system prompt that makes the model answer **only in Morse code** (dots, dashes, and slashes). Start with the raw template.

Tip: small models often need an example. Combine the system prompt with a one-shot example, as in Part 2 of 4.01.

In [ ]:
# TODO: write your system prompt
morse_prompt = """<|start_header_id|>system<|end_header_id|>

You answer only in Morse code. Use dots (.) and dashes (-), a space between letters, and a slash (/) between words. Never write normal letters.<|eot_id|><|start_header_id|>user<|end_header_id|>

Hi<|eot_id|><|start_header_id|>assistant<|end_header_id|>

.... ..<|eot_id|><|start_header_id|>user<|end_header_id|>

Hello, how are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(morse_prompt, model=CHAT_MODEL, max_tokens=100, newlines=True)

**Second task: summaries in a fixed format.** Write a system prompt that makes the model summarize a text in **one sentence of at most 15 words**, starting with `Summary:`.

No automatic score this time: read the outputs and judge them yourself. Does the model keep the format for all three texts?

In [ ]:
# three short (fictional) news texts
TEXTS = [
    "The city council of Paderborn has approved a new plan for bicycle lanes. Over the next three years, 25 kilometres "
    "of new lanes will be built, mainly on roads leading to the university and the main station. The project will cost "
    "about 12 million euros, half of which will be paid by the federal state. Local shop owners worry about losing "
    "parking spaces, while student groups welcomed the decision.",
    "A small software company from Hamburg has released a free app that helps farmers detect plant diseases. Farmers "
    "take a photo of a leaf, and the app suggests what the disease might be and how to treat it. The app works offline, "
    "which is important in rural areas with poor mobile coverage. In the first month, more than 40,000 people downloaded it.",
    "Researchers found that people who take short walks after meals have lower blood sugar levels than people who sit "
    "down. In the study, 30 participants walked for ten minutes after lunch and dinner for two weeks. The effect was "
    "strongest in the first hour after eating. The researchers say that even a few minutes of light movement can help.",
]

In [ ]:
# TODO: write your system prompt for the summaries
# "Always start ..." is not enough for this small model; one example (one-shot) makes it keep the format
summary_template = """<|start_header_id|>system<|end_header_id|>

Summarize the user's text in one sentence of at most 15 words. Always start your answer with "Summary:". Write nothing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

The new library in Bielefeld opens on Monday. It has 200,000 books, a cafe, and rooms for group work. Entry is free for everyone.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Summary: Bielefeld's new library with 200,000 books opens on Monday, free for everyone.<|eot_id|><|start_header_id|>user<|end_header_id|>

{text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
for text in TEXTS:
    response = generate(summary_template.replace("{text}", text), max_tokens=60)
    print("-", response.response.strip())

### The easy way

With the API, the system prompt is the first message in the list:

In [ ]:
messages = [
    {"role": "system", "content": "You are an AI chatbot giving sarcastic answers."},
    {"role": "user", "content": "How are you?"},
]
response = olm.chat(model=CHAT_MODEL, messages=messages, options={"temperature": 0, "seed": 42})
print(response.message.content)
what_the_model_sees(CHAT_MODEL, messages)

Compare with your hand-written text: Ollama puts our system prompt after the `Cutting Knowledge Date` line, in the same `system` block.

In a message history, the messages should alternate between `user` and `assistant`:

In [ ]:
messages = [
    {"role": "system", "content": "You are an AI chatbot giving sarcastic answers."},
    {"role": "user", "content": "How are you?"},
    {"role": "assistant", "content": "I looooove being an AI chatbot! Always being asked the same questions over and over again. It's the best!"},
    {"role": "user", "content": "What do you love the most?"},
]
response = olm.chat(model=CHAT_MODEL, messages=messages, options={"temperature": 0, "seed": 42})
print(response.message.content)

In [ ]:
# TODO: your Morse code system prompt with olm.chat()
messages = [
    {"role": "system", "content": "You answer only in Morse code. Use dots (.) and dashes (-), a space between letters, and a slash (/) between words. Never write normal letters."},
    {"role": "user", "content": "Hi"},  # one-shot example as an earlier user and assistant turn
    {"role": "assistant", "content": ".... .."},
    {"role": "user", "content": "Hello, how are you?"},
]
response = olm.chat(model=CHAT_MODEL, messages=messages, options={"temperature": 0, "seed": 42})
print(response.message.content)

## Part 3: From Chain-of-Thought to Reasoning Models

### A new model

We need a new model: [Qwen3.5 2B](https://ollama.com/library/qwen3.5).

The next cell restarts the Ollama server. This unloads the Llama models and frees GPU memory.

* **Colab / Linux:** optional on a T4 GPU, where both models fit side by side.
* **macOS / Windows with the Ollama app:** skip that cell.

In [ ]:
# stop the Ollama server (this unloads all models from the GPU memory)
!pkill -f "ollama serve"
time.sleep(2)

# start it again, exactly as in Step 2
ollama_log = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
time.sleep(5)

In [ ]:
# about 2.7 GB, the download will take a moment
!ollama pull qwen3.5:2b

In [ ]:
REASONING_MODEL = "qwen3.5:2b"

### From chain-of-thought to reasoning models

**In short: reasoning models write their reasoning on their own, before the answer.**

* **In 4.01**, the model reasoned because our examples showed it how.
* **Reasoning models** learned this during training. On problems where the answer can be checked (math, code), they were rewarded for correct answers, and they learned to write a long reasoning text first ([DeepSeek-AI, 2025](https://arxiv.org/abs/2501.12948)).
* You need neither examples nor "think step by step".

The reasoning has its own place in the template: between `<think>` and `</think>`.

### Excursion: where to find a model's template

Every model family has its own template. Ollama's `show()` returns it. Here is the template of our Llama chat model, written in Go's template language:

In [ ]:
print(olm.show(CHAT_MODEL).template)

Can you find the parts from Part 1?

* `{{ .System }}` and `{{ .Content }}` are placeholders that Ollama fills with your messages.
* `range ... .Messages` is a loop over all messages.

Now the template of Qwen3.5:

In [ ]:
print(olm.show(REASONING_MODEL).template)

Only `{{ .Prompt }}`? For newer models, Ollama builds the text with a *renderer* inside Ollama itself, so `show()` does not reveal it.

`what_the_model_sees()` works for every model, because it asks Ollama for the finished text:

In [ ]:
what_the_model_sees(REASONING_MODEL, [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "When I was 6 my sister was half my age. Now I'm 70. How old is my sister?"},
], think=True)

A complete Qwen3.5 conversation with reasoning looks like this:

<pre>
<b><font color="#B8860B">&lt;|im_start|&gt;</font></b><b><font color="#7B3FA0">system</font></b>↵
<font color="#7B3FA0">You are a helpful assistant.</font><b><font color="#B8860B">&lt;|im_end|&gt;</font></b>↵
<b><font color="#B8860B">&lt;|im_start|&gt;</font></b><b><font color="#1F5F9E">user</font></b>↵
<font color="#1F5F9E">When I was 6 my sister was half my age. Now I'm 70. How old is my sister?</font><b><font color="#B8860B">&lt;|im_end|&gt;</font></b>↵
<b><font color="#B8860B">&lt;|im_start|&gt;</font></b><b><font color="#0B6048">assistant</font></b>↵
<b><font color="#B8860B">&lt;think&gt;</font></b>↵
<font color="#C05A00">Okay, when I was 6, my sister was 3. So she is 3 years younger ...</font>↵
<b><font color="#B8860B">&lt;/think&gt;</font></b>↵
↵
<font color="#0B6048">Your sister is 67 years old.</font><b><font color="#B8860B">&lt;|im_end|&gt;</font></b>
</pre>

<b><font color="#B8860B">■ special token</font></b> &nbsp; <b><font color="#7B3FA0">■ system prompt</font></b> &nbsp; <b><font color="#1F5F9E">■ user message</font></b> &nbsp; <b><font color="#C05A00">■ reasoning (written by the model)</font></b> &nbsp; <b><font color="#0B6048">■ assistant (written by the model)</font></b> &nbsp; ↵ = line break

The tokens are different from Llama's, but the idea is the same:

* `<|im_start|>` plus the role opens a message, `<|im_end|>` closes it.
* With thinking on, the template ends with `<think>`: the model starts writing *inside* the thinking block.

### How the model sees it

Let's write it by hand and send it raw, with the settings the developers of Qwen3.5 recommend:

* **Temperature 1.0.** With a low temperature, reasoning models tend to repeat themselves in endless loops.
* **A larger context window** (`num_ctx`) for the long reasoning.

In [ ]:
reasoning_prompt = """<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
<think>
"""
complete(reasoning_prompt, model=REASONING_MODEL, max_tokens=4096, temperature=1.0, num_ctx=8192, seed=42)

Read the text between the tags: the model writes down thoughts, checks results, and sometimes corrects itself. Then it writes `</think>`, the answer, and `<|im_end|>`.

All of it is generated token by token, and every thought becomes input for the next tokens. That costs time, so look at the number of generated tokens.

### Your turn

**Predict:** what happens if the assistant's part already contains an *empty* thinking block, `<think>\n\n</think>\n\n`?

In [ ]:
no_thinking_prompt = """<|im_start|>user
When I was 6 my sister was half my age. Now I'm 70. How old is my sister?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
complete(no_thinking_prompt, model=REASONING_MODEL, max_tokens=500, temperature=1.0)

<details>
<summary><b>Explanation</b></summary>

The thinking block is already closed, so the most likely continuation is the answer itself. The model skips the reasoning. The template does exactly this when you switch thinking off.
</details>

**Your turn, continued: thinking on or off?** The word problems from 4.01 again.

* The two templates differ only in the thinking block.
* The prompt asks for `The answer is` at the end, so that `score()` finds the answer.
* With thinking on, we only use the first three problems, because every answer takes a few hundred tokens.

**Predict first:** which version scores better? How many tokens does each generate?

**Then try:** add `Think step by step.` to the thinking-off template. Does it help as much as thinking?

In [ ]:
# six word problems with their answers
WORD_PROBLEMS = [
    ("A baker makes 4 trays with 6 cookies on each tray. He sells 10 cookies. How many cookies are left?", "14"),
    ("Lisa has 3 boxes with 5 pencils each. She gives 4 pencils to her brother. How many pencils does she have now?", "11"),
    ("A bus has 20 passengers. At the first stop, 7 get off and 3 get on. How many passengers are on the bus now?", "16"),
    ("Tom reads 8 pages every day. How many pages does he read in one week?", "56"),
    ("A shirt costs 15 euros. Anna buys 2 shirts and pays with a 50 euro note. How much change does she get?", "20"),
    ("There are 12 eggs in a carton. Mia uses 5 eggs for a cake and 2 for breakfast. How many eggs are left?", "5"),
]

In [ ]:
thinking_off = """<|im_start|>user
{text} End your answer with 'The answer is' and the number.<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
score(thinking_off, WORD_PROBLEMS, model=REASONING_MODEL, max_tokens=500, temperature=1.0)

In [ ]:
thinking_on = """<|im_start|>user
{text} End your answer with 'The answer is' and the number.<|im_end|>
<|im_start|>assistant
<think>
"""
score(thinking_on, WORD_PROBLEMS[:3], model=REASONING_MODEL, max_tokens=4096, temperature=1.0, num_ctx=8192)

### The easy way

Ollama's chat API separates the two parts: the reasoning is in `message.thinking`, the answer in `message.content`.

In [ ]:
messages = [{"role": "user", "content": "When I was 6 my sister was half my age. Now I'm 70. How old is my sister?"}]

# no temperature here: Ollama uses the settings recommended for Qwen3.5, including temperature 1.0
response = olm.chat(model=REASONING_MODEL, messages=messages, think=True, options={"num_ctx": 8192})
print(highlight(response.message.thinking, ORANGE))
print("-" * 40)
print(response.message.content)

How does the reasoning go back into the model in the next turn? We send an earlier answer *with* its thinking and a follow-up question:

In [ ]:
what_the_model_sees(REASONING_MODEL, [
    {"role": "user", "content": "What is 12 x 12?"},
    {"role": "assistant", "thinking": "12 x 12 = 12 x 10 + 12 x 2 = 120 + 24 = 144.", "content": "144."},
    {"role": "user", "content": "And 13 x 13?"},
], think=True)

Look for the earlier answer: its thinking block is gone. The Qwen3.5 template keeps only the final answers of earlier turns. That saves a lot of tokens.

Other model families handle this differently, which is one more reason to look at the template.

## Reality Check: Why It Is Called Prompt *Engineering*

**In short: prompting is no exact science. What works depends on the model, the task, the wording, even the formatting.**

* You've seen it yourself: same model, same examples, different prompts, different scores.
* Changing only the format (separators, spacing, capitalization) moved the accuracy of one model on one task by up to 76 percentage points ([Sclar et al., 2024](https://arxiv.org/abs/2310.11324)).
* *The Prompt Report* lists dozens of techniques ([Schulhoff et al., 2024](https://arxiv.org/abs/2406.06608)). Many stop working with the next model generation.

That's why prompting tricks aren't the focus of this course. What lasts is what you learned today: **how a prompt template is structured, what the model actually sees, and how it continues it.** You need this again for structured outputs, tool calling, agents, and fine-tuning.

## Questions?

Anything unclear about templates, special tokens, roles, system prompts, or thinking? Ask now: the following sessions build on it.

## What You Learned

* A chat model still sees **one sequence of tokens**. The **chat template** is formatting around it.
* **Special tokens** mark where a message starts and ends and who is speaking.
* The text ends with an open assistant header. The model stops at its end-of-turn token.
* The model **does not know that it is the assistant**. It writes whatever role is open.
* The **system prompt** is the first block of the template. Templates can add text that nobody wrote.
* **Reasoning models** write their reasoning between `<think>` and `</think>`. The template decides whether thinking is on.
* **Every model family has its own template.** Chat APIs build it from your messages.
* Measure, don't guess: test templates on examples with known answers.

Next: **4.03 - Prompting with the OpenAI-Compatible API**. All of this with the API that almost every model provider offers.